# 0 - Collect Genomic Data

**Purpose**: Collect all relevant genomic data files from the Hartwig WGS data request into
per-tool CSV files for downstream analysis. Handles both standard and low-quality sample
directories.

Files collected:
- **PURPLE**: `purity`, `qc` — purity estimation and QC flags
- **CIDER**: `cider.vdj` — V(D)J CDR3 sequences
- **COBALT**: GC-corrected median read depth
- **AMBER**: BAF-based median depth (sensitivity analysis for depth normalisation)

| | |
|---|---|
| **Inputs** | `SOM_DIR/<id>/purple/`, `SOM_DIR/<id>/cider/`, `SOM_DIR/<id>/cobalt/`, `SOM_DIR/<id>/amber/` |
| **Outputs** | `PURPLE_DIR/purity.csv`, `qc.csv`; `CIDER_DIR/cider.vdj.csv`; `COBALT_DIR/depths.csv`; `AMBER_DIR/amber_depths.csv` |
| **Run before** | `1_cider.ipynb`, `4_base.ipynb` |

In [ ]:
source('../helpers/map.r')
source('../helpers/shortcuts.r')
source('../helpers/config.r')
library(parallel)
library(data.table)
library(dplyr)
library(tidyr)

n_cores <- min(32, max(1, detectCores() - 1))
message('Using ', n_cores, ' cores')

# 1 - Define samples and file targets

In [ ]:
# Collect all low-quality samples plus standard-only samples.
# LQ takes priority: samples in both directories are collected from SOM_DIR_LQ.
lq_patients       <- list.files(SOM_DIR_LQ)
standard_patients <- setdiff(list.files(SOM_DIR), lq_patients)
patients          <- c(standard_patients, lq_patients)

# Map each sample to its source directory
patient_dirs <- setNames(
  c(rep(SOM_DIR,    length(standard_patients)),
    rep(SOM_DIR_LQ, length(lq_patients))),
  c(standard_patients, lq_patients)
)

message('Samples: ', length(standard_patients), ' standard-only + ',
        length(lq_patients), ' low-quality = ', length(patients), ' total')

# Files to collect per tool
# Format: list(tool = c(file_types), output_dir = DIR)
collect_spec <- list(
  list(tool = 'purple', files = c('purity', 'qc'), out_dir = PURPLE_DIR, batch_size = 500),
  list(tool = 'cider',  files = c('cider.vdj'),    out_dir = CIDER_DIR,  batch_size = 100)
)

# Ensure output directories exist
for (spec in collect_spec) dir.create(spec$out_dir, showWarnings = FALSE, recursive = TRUE)
dir.create(COBALT_DIR, showWarnings = FALSE, recursive = TRUE)

# 2 - Collect PURPLE and CIDER tables

In [ ]:
# Resolve the file path for a given sample and file type directly from
# the sample's source directory (SOM_DIR or SOM_DIR_LQ), avoiding fragile
# string substitution on get_fp() paths.
resolve_path <- function(sample_id, file_type) {
  d <- patient_dirs[[sample_id]]
  i <- sample_id
  switch(file_type,
    "purity"    = paste0(d, i, "/purple/", i, ".purple.purity.tsv"),
    "qc"        = paste0(d, i, "/purple/", i, ".purple.qc"),
    "cider.vdj"        = paste0(d, i, "/cider/",  i, ".cider.vdj.tsv.gz"),
    "cobalt.gc.median" = paste0(d, i, "/cobalt/", i, ".cobalt.gc.median.tsv"),
    "amber.baf"        = paste0(d, i, "/amber/",  i, ".amber.baf.tsv.gz"),
    stop("Unknown file type: '", file_type, "'")
  )
}

# Read one file for one sample; returns NULL on any error.
# QC files are in key-value format and require special handling:
# they are read without headers and pivoted wide (one row per sample).
read_one <- function(sample_id, file_type) {
  file_path <- resolve_path(sample_id, file_type)
  if (!file.exists(file_path)) return(NULL)

  if (file_type == "qc") {
    tryCatch({
      fread(file_path, header = FALSE, col.names = c("field", "value")) |>
        pivot_wider(names_from = field, values_from = value) |>
        mutate(sampleId = sample_id) |>
        relocate(sampleId)
    }, error = function(e) NULL)
  } else {
    safe_read(file_path, sample_id)
  }
}

for (spec in collect_spec) {
  for (file_type in spec$files) {
    out_file <- paste0(spec$out_dir, file_type, '.csv')
    if (file.exists(out_file)) file.remove(out_file)

    message('\nCollecting [', spec$tool, '] ', file_type, ' ...')

    # Process in batches — smaller for large files (VDJ) to manage memory
    batch_size <- if (!is.null(spec$batch_size)) spec$batch_size else 500
    batches    <- split(patients, ceiling(seq_along(patients) / batch_size))
    n_batches <- length(batches)

    for (b in seq_along(batches)) {
      message(sprintf('  Batch %d/%d (size %d)', b, n_batches, batch_size)); flush.console()
      results <- mclapply(batches[[b]], read_one, file_type = file_type,
                          mc.cores = n_cores)
      out <- data.table::rbindlist(Filter(Negate(is.null), results), fill = TRUE)
      if (nrow(out) > 0)
        fwrite(out, out_file, append = file.exists(out_file))
      gc()
    }
    message('  Done: ', out_file)
  }
}

# 3 - Collect COBALT median read depths

In [ ]:
# GC-corrected median depth — one row per sample
# Used in 4_base.ipynb to normalise CIDER Score for sequencing depth
message('\nCollecting COBALT median depths ...')

total_depth <- length(patients)
counter_env <- new.env(parent = emptyenv())
counter_env$n <- 0L

depths_tmp <- mclapply(patients, function(k) {
  counter_env$n <- counter_env$n + 1L
  if (counter_env$n %% 100 == 0 || counter_env$n == total_depth)
    message(sprintf("  [%d/%d] samples processed", counter_env$n, total_depth))

  file_path <- resolve_path(k, "cobalt.gc.median")
  if (!file.exists(file_path)) return(NULL)
  tryCatch(
    fread(file_path) %>% slice(1) %>% transmute(sampleId = k, sampleMedianDepth = sampleMedian),
    error = function(e) NULL
  )
}, mc.cores = n_cores)

depths <- data.table::rbindlist(Filter(Negate(is.null), depths_tmp))
message('Depths collected for ', nrow(depths), ' samples')

fwrite(depths, paste0(COBALT_DIR, 'depths.csv'))
message('Saved: ', paste0(COBALT_DIR, 'depths.csv'))

# 4 - Collect AMBER median depths

Computes median read depth at heterozygous germline SNP positions from AMBER BAF files.
This provides a diploid-anchored depth estimate as an alternative to COBALT genome-wide median.
Used for sensitivity analysis of the CIDER Score depth normalisation.

In [ ]:
message('\nCollecting AMBER median depths ...')

amber_tmp <- mclapply(patients, function(k) {
  file_path <- resolve_path(k, "amber.baf")
  if (!file.exists(file_path)) return(NULL)
  tryCatch(
    fread(file_path, select = "tumorDepth", nThread = 1L) |>
      summarise(amberMedianDepth = median(tumorDepth, na.rm = TRUE)) |>
      transmute(sampleId = k, amberMedianDepth),
    error = function(e) NULL
  )
}, mc.cores = n_cores)

amber_depths <- data.table::rbindlist(Filter(Negate(is.null), amber_tmp))
message('AMBER depths collected for ', nrow(amber_depths), ' samples')

dir.create(AMBER_DIR, showWarnings = FALSE, recursive = TRUE)
fwrite(amber_depths, paste0(AMBER_DIR, 'amber_depths.csv'))
message('Saved: ', paste0(AMBER_DIR, 'amber_depths.csv'))